In [2]:
import pandas as pd

In [11]:
df_kalpi_address = pd.read_csv("../data/kalpi_address.csv")
columns = df_kalpi_address.columns
map_column_to_index = {
    "city_code": 2,
    "city_name": 3,
    "kalpi_code": 4,
    "kalpi_address": 6,
}
df_kalpi_address[[columns[map_column_to_index["kalpi_address"]], columns[map_column_to_index["city_name"]]]].head()

# df_kalpi_address["kalpi_full_address"] = (
#     df_kalpi_address[columns[map_column_to_index["kalpi_address"]]]
#     + ", "
#     + df_kalpi_address[columns[map_column_to_index["city_name"]]]
# )

# df_kalpi_address.to_csv("../data/kalpi_address.csv", index=False, encoding="utf-8")

,כתובת קלפי,שם ישוב בחירות
0,"מעגלי הרי""ם לוין,27",ירושלים
1,"שמואל הנביא,85",ירושלים
2,"מגן האלף,1",ירושלים
3,"פישל אהרן,29",ירושלים
4,"שבטי ישראל,27",ירושלים


In [6]:
import importlib
import sys
from pathlib import Path
sys.path.append('/workspaces/kartokalpi')
import research.geocoding
importlib.reload(research.geocoding)
from research.geocoding import GeocodingService

geo_cache_path = Path("./geocoding_cache.json")
geocoding_service = GeocodingService(geo_cache_path, enable_remote=True)

In [35]:
import tqdm
for row in tqdm.tqdm(df_kalpi_address.itertuples(), total=len(df_kalpi_address)):
    city_name = row[map_column_to_index["city_name"] + 1]
    kalpi_address = row[map_column_to_index["kalpi_address"] + 1]
    address = f"{kalpi_address},{city_name}"
    coords = await geocoding_service.get_coordinates(f"{address}")
    # print(f"{address} -> {coords}")

100%|██████████| 11547/11547 [11:59<00:00, 16.04it/s] 


In [ ]:
import asyncio

async def geocode_all_addresses():
    addresses = [
        f"{row[map_column_to_index['kalpi_address'] + 1]},{row[map_column_to_index['city_name'] + 1]}"
        for row in df_kalpi_address.itertuples()
    ]
    
    # Run all geocoding tasks concurrently
    coords = await asyncio.gather(*[geocoding_service.get_coordinates(addr) for addr in addresses])
    return coords

coords = await geocode_all_addresses()


/workspaces/kartokalpi/research/geocoding.py:50: RuntimeWarning: coroutine 'GeocodingService.get_coordinates' was never awaited
  serialized = {k: [v[0], v[1]] for k, v in self.cache.items()}


[(31.8033163, 35.2167565),
 (31.7944558, 35.2207062),
 (31.7948537, 35.2230044),
 None,
 (31.7841826, 35.2247057),
 (31.7948537, 35.2230044),
 (31.795117, 35.2248962),
 (31.7948537, 35.2230044),
 (31.7948537, 35.2230044),
 (31.7948537, 35.2230044),
 (31.7937506, 35.2246553),
 (31.7937506, 35.2246553),
 (31.7937506, 35.2246553),
 (31.7937506, 35.2246553),
 (31.7823563, 35.2246527),
 (31.784326, 35.2225869),
 (31.784326, 35.2225869),
 (31.7841826, 35.2247057),
 (31.7948537, 35.2230044),
 (31.7854221, 35.2193176),
 (31.7948537, 35.2230044),
 (31.7944558, 35.2207062),
 (31.7944558, 35.2207062),
 (31.7948537, 35.2230044),
 (31.7948537, 35.2230044),
 (31.795117, 35.2248962),
 (31.7966028, 35.216598),
 (31.7966028, 35.216598),
 (31.7948537, 35.2230044),
 (31.7919961, 35.2119799),
 (31.8152653, 35.2013819),
 (31.7937506, 35.2246553),
 (31.7607473, 35.1981136),
 (31.7890274, 35.2103024),
 (31.7203192, 35.2278325),
 (31.771169, 35.2111188),
 (31.8129538, 35.2173121),
 (31.7948537, 35.2230044),
 

In [57]:
none_count = coords.count(None)
print(f"Number of None values in coords: {none_count} out of {len(coords)}")


Number of None values in coords: 2588 out of 11547


In [59]:
df_kalpi_address["coordinates"] = coords
df_kalpi_address.to_csv("../data/kalpi_address_with_coords.csv", index=False, encoding="utf-8-sig")